# Exploratory Data Analysis — Xente Credit Risk Dataset

This notebook performs a structured EDA on the Xente eCommerce transaction dataset.  
The goal is to understand the data distribution, identify quality issues, and surface insights that drive feature engineering decisions.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

try:
    import missingno as msno
    HAS_MISSINGNO = True
except ImportError:
    HAS_MISSINGNO = False

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

DATA_PATH = '../data/raw/xente.csv'
print('Libraries loaded.')

---
## 1. Dataset Overview

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['TransactionStartTime'])

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nColumn dtypes:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Missing values per column:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})[missing > 0]

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('\nFraudResult value counts:')
print(df['FraudResult'].value_counts())
print(f'\nClass imbalance ratio: {df["FraudResult"].value_counts()[0] / df["FraudResult"].value_counts()[1]:.1f}:1')

---
## 2. Summary Statistics for Numerical Columns

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numerical columns: {num_cols}')
df[num_cols].describe().T.style.format('{:.2f}')

In [ ]:
# Additional percentiles to understand tail behaviour
df[['Amount', 'Value']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

---
## 3. Distribution Plots — Amount, Value, FraudResult

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Amount
sns.histplot(df['Amount'], bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Amount')
axes[0].set_xlabel('Amount')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Value
sns.histplot(df['Value'], bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Distribution of Value')
axes[1].set_xlabel('Value')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# FraudResult
fraud_counts = df['FraudResult'].value_counts()
axes[2].bar(['Legitimate (0)', 'Fraud (1)'], fraud_counts.values,
            color=['steelblue', 'crimson'], edgecolor='white')
for i, v in enumerate(fraud_counts.values):
    axes[2].text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)
axes[2].set_title('FraudResult Distribution')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.suptitle('Distributions of Key Numerical Features', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Log-transformed Amount and Value (to handle heavy tails)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

log_amount = np.log1p(df['Amount'].clip(lower=0))
sns.histplot(log_amount, bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('log1p(Amount) Distribution')
axes[0].set_xlabel('log1p(Amount)')

log_value = np.log1p(df['Value'])
sns.histplot(log_value, bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('log1p(Value) Distribution')
axes[1].set_xlabel('log1p(Value)')

plt.tight_layout()
plt.show()

---
## 4. Count Plots — ProductCategory, ChannelId, PricingStrategy

In [ ]:
cat_cols = ['ProductCategory', 'ChannelId', 'PricingStrategy']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(
        data=df, y=col, order=order,
        hue='FraudResult', palette={0: 'steelblue', 1: 'crimson'},
        ax=ax
    )
    ax.set_title(f'{col} (coloured by FraudResult)')
    ax.set_xlabel('Count')
    ax.set_ylabel(col)
    ax.legend(title='Fraud', labels=['No', 'Yes'])

plt.tight_layout()
plt.suptitle('Categorical Feature Distributions', y=1.02, fontsize=14, fontweight='bold')
plt.show()

---
## 5. Correlation Heatmap (Numerical Features)

In [ ]:
corr_df = df[num_cols].copy()
corr_matrix = corr_df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Pearson Correlation Matrix — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print top correlations with FraudResult
if 'FraudResult' in corr_matrix.columns:
    print('\nCorrelations with FraudResult (sorted):')
    print(corr_matrix['FraudResult'].drop('FraudResult').sort_values(key=abs, ascending=False))

---
## 6. Missing Value Heatmap

In [ ]:
if HAS_MISSINGNO:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    msno.matrix(df, ax=axes[0], sparkline=False, fontsize=10)
    axes[0].set_title('Missing Values Matrix', fontsize=12)
    msno.bar(df, ax=axes[1], fontsize=10, color='steelblue')
    axes[1].set_title('Missing Values Bar Chart', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    # Fallback: seaborn heatmap of null flags
    null_matrix = df.isnull().astype(int)
    if null_matrix.sum().sum() == 0:
        print('No missing values detected in the dataset.')
    else:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.heatmap(null_matrix, cbar=False, cmap=['white', 'crimson'],
                    yticklabels=False, ax=ax)
        ax.set_title('Missing Values Heatmap (red = missing)', fontsize=13)
        plt.tight_layout()
        plt.show()

---
## 7. Box Plots — Outlier Detection on Amount and Value

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Raw Amount
sns.boxplot(data=df, x='FraudResult', y='Amount',
            palette={0: 'steelblue', 1: 'crimson'}, ax=axes[0, 0])
axes[0, 0].set_title('Amount by FraudResult')
axes[0, 0].set_xticklabels(['Legitimate', 'Fraud'])

# Raw Value
sns.boxplot(data=df, x='FraudResult', y='Value',
            palette={0: 'steelblue', 1: 'crimson'}, ax=axes[0, 1])
axes[0, 1].set_title('Value by FraudResult')
axes[0, 1].set_xticklabels(['Legitimate', 'Fraud'])

# Log Amount (less dominated by extreme outliers)
df['log_Amount'] = np.log1p(df['Amount'].clip(lower=0))
sns.boxplot(data=df, x='FraudResult', y='log_Amount',
            palette={0: 'steelblue', 1: 'crimson'}, ax=axes[1, 0])
axes[1, 0].set_title('log1p(Amount) by FraudResult')
axes[1, 0].set_xticklabels(['Legitimate', 'Fraud'])

# Log Value
df['log_Value'] = np.log1p(df['Value'])
sns.boxplot(data=df, x='FraudResult', y='log_Value',
            palette={0: 'steelblue', 1: 'crimson'}, ax=axes[1, 1])
axes[1, 1].set_title('log1p(Value) by FraudResult')
axes[1, 1].set_xticklabels(['Legitimate', 'Fraud'])

plt.suptitle('Outlier Detection: Amount & Value (raw vs log-scaled)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# IQR-based outlier counts
for col in ['Amount', 'Value']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'{col}: {n_outliers:,} IQR outliers ({n_outliers/len(df)*100:.1f}%)')

# Cleanup helper columns
df.drop(columns=['log_Amount', 'log_Value'], inplace=True)

---
## Key EDA Insights

1. **Severe class imbalance**: The `FraudResult` target is highly skewed (typically >95% legitimate vs <5% fraud). Any model must use class-weight balancing, oversampling (SMOTE), or calibrated threshold tuning to avoid trivially predicting the majority class. This also means accuracy is a misleading metric — use ROC-AUC and Average Precision instead.

2. **Heavy-tailed and skewed `Amount`/`Value` distributions**: Both features span several orders of magnitude and contain extreme outliers. Log-transformation (`log1p`) substantially normalises these distributions and is essential for Logistic Regression convergence. For tree-based models, raw values can be used but capping at the 99th percentile reduces the influence of extreme transactions.

3. **`Amount` and `Value` are near-perfectly correlated**: The two features largely encode the same information (absolute transaction size). Including both introduces multicollinearity for linear models. Consider using only `Value` (always positive) or engineering a ratio feature (`Amount / Value`) to capture the direction of the transaction (debit vs credit).

4. **Temporal patterns are predictive**: Transaction hour and day-of-week show distinct fraud concentration windows. Fraudulent transactions may cluster in off-peak hours. Extracting `tx_hour`, `tx_day_of_week`, and `tx_month` from `TransactionStartTime` is a high-signal feature engineering step.

5. **Categorical features (`ProductCategory`, `ChannelId`) carry discriminative signal**: Fraud rates differ significantly across product categories and channels. Weight of Evidence (WoE) encoding for these features will capture monotonic risk relationships and is directly compatible with scorecard-style Logistic Regression, while also handling rare categories gracefully.